# Experiment: Benchmark All Tools

What this notebook teaches:
- Run the repository benchmark script from Colab or local runtime.
- Generate `benchmark.csv`, `benchmark.md`, and `environment.json` artifacts.
- Compare measured tools and keep unmeasured tools explicitly marked.


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/sumeyye-agac/human-pose-estimation-experiments.git"
REPO_DIR = Path("/content/human-pose-estimation-experiments")

if "google.colab" in sys.modules:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    raise RuntimeError("Run this notebook from the repository root.")

if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

print(f"Using repo root: {repo_root}")


In [ ]:
def pip_install(*packages: str) -> None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

pip_install("numpy", "pandas", "matplotlib")


In [ ]:
import subprocess

subprocess.run(
    [
        sys.executable,
        "scripts/run_benchmarks.py",
        "--tool",
        "all",
        "--frames",
        "60",
        "--warmup",
        "12",
        "--repeat",
        "1",
    ],
    check=True,
)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

benchmark_df = pd.read_csv(repo_root / "results" / "benchmark.csv")
benchmark_df


In [ ]:
measured = benchmark_df[benchmark_df["status"] == "measured"].copy()

if measured.empty:
    print("No measured rows yet. Run tool-specific setup notebooks first.")
else:
    plt.figure(figsize=(6, 3.5))
    plt.bar(measured["tool"], measured["fps"])
    plt.ylabel("FPS")
    plt.title("Measured benchmark snapshot")
    plt.tight_layout()
    figure_path = repo_root / "assets" / "generated" / "benchmark_fps.png"
    figure_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(figure_path, dpi=140)
    plt.show()
    print("Saved", figure_path)
